# 01 - Redes Neuronales desde Cero

En este notebook construiremos una red neuronal **completamente desde cero** usando solo NumPy.
El objetivo es entender los fundamentos matemáticos antes de usar frameworks como PyTorch o TensorFlow.

## Contenido
1. Neurona simple (perceptrón)
2. Funciones de activación
3. Forward propagation
4. Función de pérdida (loss)
5. Backpropagation (la magia del aprendizaje)
6. Entrenamiento completo en un dataset real
7. Comparación con PyTorch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

## 1. La Neurona: Unidad Básica

Una neurona artificial calcula:

$$y = f(\mathbf{w} \cdot \mathbf{x} + b)$$

Donde:
- $\mathbf{x}$ = entradas (features)
- $\mathbf{w}$ = pesos (weights) — qué tan importante es cada entrada
- $b$ = sesgo (bias) — umbral de activación
- $f$ = función de activación — introduce no-linealidad

In [ ]:
class Neuron:
    """Una sola neurona con pesos, bias y activación."""
    
    def __init__(self, n_inputs):
        # Inicialización Xavier: pesos pequeños y aleatorios
        self.weights = np.random.randn(n_inputs) * np.sqrt(2.0 / n_inputs)
        self.bias = 0.0
    
    def forward(self, x):
        # Producto punto + bias
        z = np.dot(self.weights, x) + self.bias
        # Activación sigmoid: convierte cualquier número a [0, 1]
        return 1 / (1 + np.exp(-z))

# Ejemplo: neurona con 3 entradas
neuron = Neuron(3)
x = np.array([1.0, 0.5, -0.3])
output = neuron.forward(x)
print(f"Entrada: {x}")
print(f"Pesos: {neuron.weights}")
print(f"Salida: {output:.4f}")

## 2. Funciones de Activación

Sin activación, una red neuronal sería solo una regresión lineal (por más capas que tenga).
Las funciones de activación introducen **no-linealidad**, permitiendo aprender patrones complejos.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z) ** 2

# Visualización
z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, func, deriv) in zip(axes, [
    ('Sigmoid', sigmoid, sigmoid_derivative),
    ('ReLU', relu, relu_derivative),
    ('Tanh', tanh, tanh_derivative),
]):
    ax.plot(z, func(z), label=name, linewidth=2)
    ax.plot(z, deriv(z), '--', label=f'{name} derivada', alpha=0.7)
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Sigmoid: salida [0,1] — útil para probabilidades")
print("ReLU: la más usada — simple, eficiente, evita vanishing gradients")
print("Tanh: salida [-1,1] — centrada en cero, mejor que sigmoid en capas ocultas")

## 3. Red Neuronal Completa desde Cero

Vamos a construir una red con:
- Capa de entrada: n features
- Capa oculta: neuronas con ReLU
- Capa de salida: sigmoid (clasificación binaria) o softmax (multiclase)

In [ ]:
class NeuralNetwork:
    """Red neuronal feedforward desde cero."""
    
    def __init__(self, layer_sizes):
        """
        Args:
            layer_sizes: lista [input, hidden1, hidden2, ..., output]
                         Ejemplo: [784, 128, 64, 10] para MNIST
        """
        self.n_layers = len(layer_sizes) - 1
        self.weights = []
        self.biases = []
        
        # Inicialización He (buena para ReLU)
        for i in range(self.n_layers):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
        
        print(f"Red creada: {' → '.join(str(s) for s in layer_sizes)}")
        total_params = sum(w.size + b.size for w, b in zip(self.weights, self.biases))
        print(f"Total parámetros: {total_params:,}")
    
    def _softmax(self, z):
        """Softmax estable numéricamente."""
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)
    
    def forward(self, X):
        """Forward pass — calcula la salida de la red."""
        self.activations = [X]  # Guardamos para backprop
        self.z_values = []
        
        current = X
        for i in range(self.n_layers):
            z = self.weights[i] @ current + self.biases[i]
            self.z_values.append(z)
            
            if i == self.n_layers - 1:
                # Última capa: softmax
                current = self._softmax(z)
            else:
                # Capas ocultas: ReLU
                current = relu(z)
            
            self.activations.append(current)
        
        return current
    
    def compute_loss(self, y_pred, y_true):
        """Cross-entropy loss."""
        m = y_true.shape[1]
        # Clip para evitar log(0)
        y_pred = np.clip(y_pred, 1e-8, 1 - 1e-8)
        loss = -np.sum(y_true * np.log(y_pred)) / m
        return loss
    
    def backward(self, y_true, learning_rate=0.01):
        """Backpropagation — el algoritmo que hace que las redes aprendan.
        
        Idea clave: calcular cuánto contribuyó cada peso al error,
        y ajustarlo en la dirección que reduce el error (gradiente descendente).
        """
        m = y_true.shape[1]  # batch size
        
        # Gradiente de la última capa (softmax + cross-entropy)
        dz = self.activations[-1] - y_true
        
        for i in range(self.n_layers - 1, -1, -1):
            # Gradientes de pesos y biases
            dw = (dz @ self.activations[i].T) / m
            db = np.sum(dz, axis=1, keepdims=True) / m
            
            if i > 0:
                # Propagar gradiente a la capa anterior
                dz = (self.weights[i].T @ dz) * relu_derivative(self.z_values[i-1])
            
            # Actualizar parámetros (gradient descent)
            self.weights[i] -= learning_rate * dw
            self.biases[i] -= learning_rate * db
    
    def train(self, X, y, epochs=100, learning_rate=0.01, batch_size=64, verbose=True):
        """Entrenamiento completo con mini-batches."""
        m = X.shape[1]
        history = {'loss': [], 'accuracy': []}
        
        for epoch in range(epochs):
            # Shuffle datos
            indices = np.random.permutation(m)
            X_shuffled = X[:, indices]
            y_shuffled = y[:, indices]
            
            epoch_loss = 0
            n_batches = 0
            
            for start in range(0, m, batch_size):
                end = min(start + batch_size, m)
                X_batch = X_shuffled[:, start:end]
                y_batch = y_shuffled[:, start:end]
                
                # Forward
                y_pred = self.forward(X_batch)
                epoch_loss += self.compute_loss(y_pred, y_batch)
                n_batches += 1
                
                # Backward
                self.backward(y_batch, learning_rate)
            
            avg_loss = epoch_loss / n_batches
            
            # Calcular accuracy
            y_pred_all = self.forward(X)
            predictions = np.argmax(y_pred_all, axis=0)
            labels = np.argmax(y, axis=0)
            accuracy = np.mean(predictions == labels)
            
            history['loss'].append(avg_loss)
            history['accuracy'].append(accuracy)
            
            if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch+1:3d}/{epochs} — Loss: {avg_loss:.4f} — Accuracy: {accuracy:.4f}")
        
        return history
    
    def predict(self, X):
        """Predecir clases."""
        y_pred = self.forward(X)
        return np.argmax(y_pred, axis=0)

## 4. Entrenamiento en un Dataset Real: Dígitos

Usaremos sklearn's digits dataset (8x8 imágenes de dígitos 0-9).
Es como un mini-MNIST perfecto para aprender.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Cargar datos
digits = load_digits()
X_raw = digits.data    # (1797, 64) — 1797 imágenes de 8x8
y_raw = digits.target  # (1797,) — etiquetas 0-9

print(f"Dataset: {X_raw.shape[0]} imágenes de {int(np.sqrt(X_raw.shape[1]))}x{int(np.sqrt(X_raw.shape[1]))} píxeles")
print(f"Clases: {len(np.unique(y_raw))} (dígitos 0-9)")

# Visualizar algunas muestras
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_raw[i].reshape(8, 8), cmap='gray')
    ax.set_title(str(y_raw[i]))
    ax.axis('off')
plt.suptitle('Muestras del dataset de dígitos', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Preparar datos para nuestra red
# 1. Normalizar [0, 16] → [0, 1]
X_norm = X_raw / 16.0

# 2. Transponer: nuestra red espera (features, samples)
X = X_norm.T  # (64, 1797)

# 3. One-hot encode las etiquetas
def one_hot(y, n_classes):
    oh = np.zeros((n_classes, len(y)))
    oh[y, np.arange(len(y))] = 1
    return oh

y = one_hot(y_raw, 10)  # (10, 1797)

# 4. Split train/test
split = int(0.8 * X.shape[1])
indices = np.random.permutation(X.shape[1])
X_train, X_test = X[:, indices[:split]], X[:, indices[split:]]
y_train, y_test = y[:, indices[:split]], y[:, indices[split:]]

print(f"Train: {X_train.shape[1]} muestras")
print(f"Test: {X_test.shape[1]} muestras")

In [ ]:
# ¡Crear y entrenar la red!
# Arquitectura: 64 → 128 → 64 → 10
nn = NeuralNetwork([64, 128, 64, 10])

history = nn.train(
    X_train, y_train,
    epochs=100,
    learning_rate=0.1,
    batch_size=32,
)

In [ ]:
# Visualizar entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'])
ax1.set_title('Loss durante entrenamiento')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['accuracy'])
ax2.set_title('Accuracy durante entrenamiento')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluar en test
test_preds = nn.predict(X_test)
test_labels = np.argmax(y_test, axis=0)
test_accuracy = np.mean(test_preds == test_labels)
print(f"\nAccuracy en TEST: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"¡Nuestra red hecha desde cero clasifica dígitos con ~{test_accuracy*100:.0f}% de precisión!")

In [ ]:
# Matriz de confusión
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 5. Comparación: Lo Mismo en PyTorch

Ahora que entendemos lo que pasa internamente, veamos cómo PyTorch
hace exactamente lo mismo en muchas menos líneas (pero la magia es la misma).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Misma arquitectura: 64 → 128 → 64 → 10
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

# PyTorch calcula los gradientes automáticamente (autograd)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Convertir datos a tensores
X_train_t = torch.FloatTensor(X_train.T)  # PyTorch: (samples, features)
y_train_t = torch.LongTensor(np.argmax(y_train, axis=0))
X_test_t = torch.FloatTensor(X_test.T)
y_test_t = torch.LongTensor(np.argmax(y_test, axis=0))

# Entrenar
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()       # ← ¡Backprop automático!
    optimizer.step()      # ← Actualizar pesos
    
    if epoch % 20 == 0:
        with torch.no_grad():
            preds = model(X_test_t).argmax(dim=1)
            acc = (preds == y_test_t).float().mean()
        print(f"Epoch {epoch+1:3d} — Loss: {loss.item():.4f} — Test Acc: {acc:.4f}")

# Evaluar
with torch.no_grad():
    preds = model(X_test_t).argmax(dim=1)
    acc = (preds == y_test_t).float().mean()
print(f"\nPyTorch Test Accuracy: {acc:.4f}")

## Resumen

| Concepto | Nuestra implementación | PyTorch |
|----------|----------------------|----------|
| Forward pass | `forward()` manual con matrices | `model(x)` |
| Loss | `compute_loss()` con NumPy | `nn.CrossEntropyLoss()` |
| Backprop | `backward()` derivadas manuales | `loss.backward()` autograd |
| Actualizar pesos | Gradient descent manual | `optimizer.step()` |

**Lo esencial:**
- Una red neuronal es una función matemática parametrizada
- Forward pass: calcula la salida dada una entrada
- Loss: mide qué tan mal está la predicción
- Backpropagation: calcula cómo ajustar cada peso para reducir el error
- Gradient descent: actualiza los pesos en la dirección que reduce el error

**Siguiente notebook:** Redes Convolucionales (CNNs) — cómo las redes "ven" imágenes.